In [1]:
!pip install faster-whisper jiwer librosa soundfile tqdm num2words --quiet
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1" "onnxruntime-gpu==1.20.1" --quiet


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


# Importing libraries

In [2]:
import os
import pandas as pd
import re
from faster_whisper import WhisperModel 
from jiwer import wer
from tqdm import tqdm
from num2words import num2words #for converting numbers into text
import soundfile as sf
import torchaudio
from transformers import AutoProcessor, AutoModel
import torch

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Get the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login programmatically (no interactive prompt)
login(token=hf_token)

# Downloading dataset

In [4]:
base_path = "/kaggle/input/competitions/multilingual-speech-recognition"

train_df = pd.read_csv(f"{base_path}/train.csv")
test_df = pd.read_csv(f"{base_path}/test.csv")

print(train_df.head())
print(test_df.head())

   id            audio                                               text
0   0  audio_00000.wav                      you had quoted plutarch line.
1   1  audio_00001.wav  மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2   2  audio_00002.wav  to do his phd in engineering about four years ...
3   3  audio_00003.wav                          maybe he was not at home.
4   4  audio_00004.wav  BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...
   id            audio
0   0  audio_00000.wav
1   1  audio_00001.wav
2   2  audio_00002.wav
3   3  audio_00003.wav
4   4  audio_00004.wav


In [5]:
train_df["audio_path"] = train_df["audio"].apply(
    lambda x: f"{base_path}/competition_data/train/{x}"
)

test_df["audio_path"] = test_df["audio"].apply(
    lambda x: f"{base_path}/competition_data/test/{x}"
)

## Some analysis

### Conclusions

* There are only three languages in the train dataset -- English, Tamil, Hindi.
* The distribution of the languages is around 50:25:25 (in percentage)
* Most of the clips were less than 15 secs with maximum length a clip was 24 secs and minimum of around 4 secs
* Hindi language clips were shorter compared to the other two languages and comparatively of poor quality

In [6]:
def detect_language(text):
    has_tamil = bool(re.search(r'[\u0B80-\u0BFF]', text))
    has_hindi = bool(re.search(r'[\u0900-\u097F]', text))
    has_english = bool(re.search(r'[a-zA-Z]', text))
    
    count = sum([has_tamil, has_hindi, has_english])
    
    if count > 1:
        return "mixed"
    elif has_tamil:
        return "tamil"
    elif has_hindi:
        return "hindi"
    elif has_english:
        return "english"
    else:
        return "unknown"

In [7]:
train_df["language"] = train_df["text"].apply(detect_language)

In [8]:
lang_counts = train_df["language"].value_counts()
lang_counts

language
english    998
hindi      508
tamil      494
Name: count, dtype: int64

In [9]:
def get_duration(path):
    return sf.info(path).duration

In [10]:
durations = []

for path in tqdm(train_df["audio_path"]):
    durations.append(get_duration(path))

train_df["duration"] = durations

100%|██████████| 2000/2000 [00:11<00:00, 171.36it/s]


In [11]:
stats = train_df.groupby("language")["duration"].agg(
    ["count", "mean", "min", "max"]
)

stats

,count,mean,min,max
language,,,,
english,998,8.476284,1.380,17.210
hindi,508,5.019064,2.240,10.112
tamil,494,8.876121,1.022,24.190


In [12]:
audio_path = train_df["audio_path"].iloc[0]

info = sf.info(audio_path)


print(info)

/kaggle/input/competitions/multilingual-speech-recognition/competition_data/train/audio_00000.wav
samplerate: 48000 Hz
channels: 1
duration: 2.900 s
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]


In [13]:
train_df.head()

,id,audio,text,audio_path,language,duration
0,0,audio_00000.wav,you had quoted plutarch line.,/kaggle/input/competitions/multilingual-speech...,english,2.900
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...,/kaggle/input/competitions/multilingual-speech...,tamil,12.733
2,2,audio_00002.wav,to do his phd in engineering about four years ...,/kaggle/input/competitions/multilingual-speech...,english,5.670
3,3,audio_00003.wav,maybe he was not at home.,/kaggle/input/competitions/multilingual-speech...,english,2.880
4,4,audio_00004.wav,BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...,/kaggle/input/competitions/multilingual-speech...,english,14.385


# Models download

* The choice of model was arrived at after some initial studies and on a trial-and-error basis. Wav2vec2 was ruled out as its primary objective self-supervised learning, which is appropriate for generation. But in our case, labels were given for training dataset.
* Subsequently, I chose whisper models to pursue as it were pretrained extensively on English, Hindi and Tamil, which are the given languages for our task. I initially selected Whisper-Medium keeping in mind compute. Then I tried Whisper-Large-V3-Turbo.
* Since, the Whisper-Medium did not yield desired results, I chose the Indic-language model -- AI4Bharat's Indic-conformer-600m-multilingual

In [14]:
model_medium = WhisperModel(
    "medium",
    compute_type="float16" #float16 for gpu; int8 for cpu 
)

In [15]:
ai4_model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True
)

ai4_model = ai4_model.to("cuda")
ai4_model.eval()

config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


IndicASRModel()

In [16]:
# model_large = WhisperModel(
#     "large-v3-turbo",
#     compute_type="float16"   
# )

# Baseline scores

* Whisper-Medium gave a baseline score of 0.647.
* For Whisper large turbo, the baseline score was 0.666
* Normalization carried out includes lowercasing English alphabets and removing puncations in the predictions  

In [17]:
def normalize_text(text):
    text = text.lower()
    
    # selecting only English/ Tamil / Hindi characters
    text = re.sub(r"[^\w\s\u0B80-\u0BFF\u0900-\u097F]", "", text)
    
    text = " ".join(text.split()) # Normalizing whitespace
    
    return text

In [18]:
#Calling the normalization function
train_df["clean_text"] = train_df["text"].apply(normalize_text)
print(train_df[["text", "clean_text"]].head())

                                                text  \
0                      you had quoted plutarch line.   
1  மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...   
2  to do his phd in engineering about four years ...   
3                          maybe he was not at home.   
4  BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...   

                                          clean_text  
0                       you had quoted plutarch line  
1  மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...  
2  to do his phd in engineering about four years ago  
3                           maybe he was not at home  
4  but we didnt break his old window you know exp...  


In [19]:
def transcribe_audio(model, audio_path):
    segments, _ = model_medium.transcribe( #model_large for large model
        audio_path,
        beam_size=5,
        temperature=0 
    )
    text = " ".join([seg.text for seg in segments])
    return normalize_text(text)

In [20]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = transcribe_audio(model_medium,row["audio_path"])
    predictions.append(pred)

test_df["prediction"] = predictions


100%|██████████| 100/100 [02:10<00:00,  1.31s/it]


In [21]:
submission = test_df[["audio", "prediction"]].copy()
submission.columns = ["audio", "text"]
submission.to_csv("submission.csv", index=False)

# More normalization
* As Whisper-Medium was already pretrained sufficiently in the desired language and as the train-dataset is not that large, I decided to make the predictions align with ground truth in train dataset than fine-tuning the model.
* So, I enforced the model to predict only the languages in the train dataset as some Hindi clips were transcribed in Urdu.
* Then I converted numbers into words in English but was unable to do so for the other two languages
* There were marginal improvement in score after this.
* Subsequently, I added '.' at the end the English sentences as there were pull stops in the all-small English sentences in train dataset. This decreased WER to around 0.59 from around 0.63.

In [22]:
def map_language(lang):
    if lang in ["en"]:
        return "en"
    elif lang in ["hi", "ur"]:
        return "hi"
    else:
        return "ta"

In [23]:
def transcribe_audio(model, audio_path):
    # first pass: detect
    segments, info = model.transcribe(audio_path)
    
    lang = map_language(info.language)
    
    # second pass: enforce
    segments, _ = model.transcribe(
        audio_path,
        beam_size=5,
        temperature=0,
        language=lang
    )
    
    text = " ".join([seg.text for seg in segments])
    return text, lang

In [24]:
def normalize_numbers(text, lang):
    
    def replace(match):
        num = int(match.group())
        
        try:
            if lang == "en":
                return num2words(num)
            else:
                return str(num)  # keeping digits for now in other languages
        except:
            return str(num)
    
    return re.sub(r"\b\d+\b", replace, text)

In [25]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\u0B80-\u0BFF\u0900-\u097F]", "", text)
    text = " ".join(text.split())
    return text

In [26]:
def add_period_if_english(text):
    text = text.strip()
    
    # Check: only English letters and spaces
    if re.fullmatch(r"[a-z\s]+", text):
        if not text.endswith("."):
            return text + "."
    
    return text

In [27]:
def process_audio(model, audio_path):
    text, lang = transcribe_audio(model, audio_path)
    
    text = normalize_numbers(text, lang)
    text = normalize_text(text)
    text = add_period_if_english(text)
    
    return text

In [28]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = process_audio(model_medium, row["audio_path"])
    predictions.append(pred)

100%|██████████| 100/100 [02:44<00:00,  1.64s/it]


In [29]:
test_df["prediction"] = predictions
submission = test_df[["audio", "prediction"]].copy()
submission.columns = ["audio", "text"]

submission.to_csv("submission.csv", index=False)

# Hybrid models

* I found Hindi and Tamil transcriptions were of poor quality and tokenization was not that good
* To improve these transcriptions, I used indic-conformer-600M-multilingual model of AI4Bharat which was pretrained 22 Indian languages including Hindi and Tamil.
* While AI4Bharat's model was used for Hindi and Tamil, Whisper-Medium was used for transcription of English.
* This simple use of multiple models improved score 0.39.
* Several permutations were done such as post-processing of generated texts, analysis of train dataset, and conclusions from these results were incorporated, but none of this decreased WER score.
* Then I came to know that Ground truth in test dataset was case sensitive. But before doing anything about it, the inference file was made case insenstive again.
* Then simple re-running of the code decreased WER to around 0.163

In [30]:
# Whisper Medium was used to detect the language
def detect_language(audio_path):
    _, info = model_medium.transcribe(audio_path, beam_size=1)
    return info.language

In [31]:
#for English along
def transcribe_whisper(audio_path):
    segments, _ = model_medium.transcribe(
        audio_path,
        beam_size=5,
        temperature=0, # Don't want it be creative 
        patience=0.8 # Wanted the beam search not to look more
    )
    return " ".join([seg.text for seg in segments]).strip()

In [32]:
def transcribe_ai4bharat(audio_path, lang):
    wav, sr = torchaudio.load(audio_path)
    
    # mono
    wav = torch.mean(wav, dim=0, keepdim=True)
    
    # resample
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        wav = resampler(wav)
    
    wav = wav.to("cuda")
    
    with torch.no_grad():
        text = ai4_model(wav, lang, "ctc")  
    
    # clean artifacts
    text = text.replace("|", " ") 
    
    return text.strip()

In [33]:
def is_english_text(text): #Identifying only English text
    if re.search(r"[\u0900-\u097F\u0B80-\u0BFF]", text):
        return False
    return bool(re.fullmatch(r"[a-zA-Z0-9\s]+", text))


def normalize_numbers(text):
    if not is_english_text(text):
        return text
    
    def replace(match):
        num = int(match.group())
        try:
            return num2words(num)
        except:
            return str(num)
    
    return re.sub(r"\b\d+\b", replace, text)


In [34]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\u0B80-\u0BFF\u0900-\u097F]", "", text)
    text = " ".join(text.split())
    return text

In [35]:
def add_period_if_english(text):
    if re.fullmatch(r"[a-z\s]+", text):
        if not text.endswith("."):
            return text + "."
    return text

In [36]:
def process_audio(audio_path):
    lang = detect_language(audio_path)
    
    if lang == "en":
        text = transcribe_whisper(audio_path)
    
    elif lang in ["hi", "ur"]:
        text = transcribe_ai4bharat(audio_path, "hi")
    
    elif lang == "ta":
        text = transcribe_ai4bharat(audio_path, "ta")
    
    else:
        text = transcribe_whisper(audio_path)
    
    # normalization
    text = normalize_numbers(text)
    text = normalize_text(text)
    text = add_period_if_english(text)
    
    return text

In [37]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = process_audio(row["audio_path"])
    predictions.append(pred)

test_df["prediction"] = predictions

100%|██████████| 100/100 [02:29<00:00,  1.49s/it]


In [38]:
submission = test_df[["audio", "prediction"]].copy()
submission.columns = ["audio", "text"]

submission.to_csv("submission.csv", index=False)